In [7]:
import duckdb
import os
import tomllib
from tqdm.notebook import tqdm
from dotenv import load_dotenv

In [3]:
# CONFIGURAÇÃO DE URI

load_dotenv()

db_host = os.getenv("DB_HOST")
db_user = os.getenv("DB_USER")
db_pass = os.getenv("DB_PASS")
db_name = os.getenv("DB_NAME")
db_port = os.getenv("DB_PORT")

DB_URI = (
    f"dbname={db_name} password={db_pass} user={db_user} host={db_host} port={db_port}"
)

In [ ]:
# CONFIGURAÇÃO DE SOURCE E TARGET DATABASE

load_dotenv()

# SOURCE
src_db_host = os.getenv("SOURCE_DB_HOST")
src_db_user = os.getenv("SOURCE_DB_USER")
src_db_pass = os.getenv("SOURCE_DB_PASS")
src_db_name = os.getenv("SOURCE_DB_NAME")
src_db_port = os.getenv("SOURCE_DB_PORT")

SOURCE_DB_URI = f"src_db_name={src_db_name} password={src_db_pass} user={src_db_user} host={src_db_host} port={src_db_port}"

# TARGET
trg_db_host = os.getenv("TARGET_DB_HOST")
trg_db_user = os.getenv("TARGET_DB_USER")
trg_db_pass = os.getenv("TARGET_DB_PASS")
trg_db_name = os.getenv("TARGET_DB_NAME")
trg_db_port = os.getenv("TARGET_DB_PORT")

TARGET_DB_URI = f"src_db_name={trg_db_name} password={trg_db_pass} user={trg_db_user} host={trg_db_host} port={trg_db_port}"

In [4]:
# CONEXAO COM O DUCKDB

con = duckdb.connect("INSTALL postgres;")
con = duckdb.connect("LOAD postgres;")

con.execute("PRAGMA enable_progress_bar;")
con.execute("PRAGMA enable_print_progress_bar;")
con.execute(f"ATTACH '{DB_URI}' AS dblocal_postgres (TYPE postgres);")

In [ ]:
con.sql(f"SELECT * FROM dblocal_postgres.base_filme_teste WHERE Ano_Lançamento = 2000;")

TítuloVARCHAR,Ano_LançamentoBIGINT,NotaDOUBLE,GêneroVARCHAR
As Fantasma Profundo,2000,4.9,Crime
Um Amor do Futuro,2000,2.5,Mistério
O Segredo de Amanhecer de Fogo,2000,3.6,Ação
O Último Mundo do Futuro,2000,5.1,Terror
O Retorno de Noite Profundo,2000,8.5,Drama
Crônicas de Amanhecer Eterna,2000,4.3,Ação
Crônicas de Guerreiro Eterna,2000,1.5,Musical
O Retorno de Rainha Renascido,2000,3.6,Ação
O Despertar de Planeta do Amanhã,2000,7.6,Mistério
A Nova Espada Supremo,2000,9.3,Crime


In [ ]:
con.sql(
    "SELECT * FROM dblocal_postgres.base_filme_teste WHERE Gênero = 'Ação' AND Nota > 7.0;"
)

TítuloVARCHAR,Ano_LançamentoBIGINT,NotaDOUBLE,GêneroVARCHAR
O Retorno de Estrela Profundo,1989,7.7,Ação
As Fantasma Sombrio,2013,9.8,Ação
A Nova Vingança de Fogo,2014,9.2,Ação
O Retorno de Planeta Dourada,2017,9.0,Ação
O Segredo de Fogo Dourada,2002,9.5,Ação
A Lenda de Guerreiro Silencioso,1968,8.7,Ação
Uma Rainha Oculta,2008,7.8,Ação
A Nova Princesa Infinito,1990,7.1,Ação
A Lenda de Guardião Sombrio,1966,7.8,Ação
As Universo Sombria,2004,9.9,Ação


In [5]:
con.sql("SELECT * FROM dblocal_postgres.base_filme_teste WHERE Ano_Lançamento > 2012;")

TítuloVARCHAR,Ano_LançamentoBIGINT,NotaDOUBLE,GêneroVARCHAR
Os Gelo Esquecido,2015,6.8,Comédia
Crônicas de Cidade Perdido,2016,3.3,Aventura
A Lenda de Luz do Passado,2021,8.7,Animação
O Segredo de Ilusão de Ouro,2017,9.3,Documentário
As Fantasma Sombrio,2013,9.8,Ação
O Despertar de Destino Eterno,2020,4.8,Fantasia
Noite em Planeta Silenciosa,2022,6.1,Aventura
Um Mistério Profunda,2022,9.1,Drama
O Amanhecer Sombria,2025,4.7,Ficção Científica
Os Anjo Perdida,2025,9.0,Faroeste


In [6]:
# CONFIGURAÇÕES DE BANCO
src_schema, trg_schema, src_nome_tabela, trg_nome_tabela = (
    "public",
    "public",
    "car_imoveis_validos_a",
    "car_imoveis_validos_a",
)

TAMANHO_LOTE = 100000

# CARREGAR DADOS DOTENV

load_dotenv()

db_host = os.getenv("DB_HOST")
db_user = os.getenv("DB_USER")
db_pass = os.getenv("DB_PASS")
db_name = os.getenv("DB_NAME")

DB_URI = f"dbname={db_host} user={db_user} password={db_pass} host={db_name}"

# SOURCE
src_db_host = os.getenv("SOURCE_DB_HOST")
src_db_user = os.getenv("SOURCE_DB_USER")
src_db_pass = os.getenv("SOURCE_DB_PASS")
src_db_name = os.getenv("SOURCE_DB_NAME")
src_db_port = os.getenv("SOURCE_DB_PORT")

SOURCE_DB_URI = f"dbname={src_db_name} password={src_db_pass} user={src_db_user} host={src_db_host} port={src_db_port}"

# TARGET
trg_db_host = os.getenv("TARGET_DB_HOST")
trg_db_user = os.getenv("TARGET_DB_USER")
trg_db_pass = os.getenv("TARGET_DB_PASS")
trg_db_name = os.getenv("TARGET_DB_NAME")
trg_db_port = os.getenv("TARGET_DB_PORT")

TARGET_DB_URI = f"dbname={trg_db_name} password={trg_db_pass} user={trg_db_user} host={trg_db_host} port={trg_db_port}"

In [ ]:
tabela_existe = """
SELECT EXISTS(
        SELECT 1
        FROM information_schema.tables
        WHERE table_schema = '{trg_schema}' AND table_name = '{trg_nome_tabela}' 
        );
"""